# Ch 19 — 양자화 입문

int8 / int4 PTQ (Post-Training Quantization) 으로 모델 메모리를 1/4 ~ 1/8 로 줄입니다.

In [ ]:
# 필요 시 설치
# !pip install torch

import torch
import torch.nn as nn
import math

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {device}')

## 1. 개념 — 비트 줄이기

| 형식 | bytes | 표현 가능 값 | 정확도 손실 |
|---|---:|---|---|
| fp32 | 4 | ±3.4×10³⁸ | 0 (기준) |
| fp16 | 2 | ±6.5×10⁴ | <1% |
| **int8** | **1** | -128 ~ 127 (256 값) | **2~5%** |
| **int4** | **0.5** | -8 ~ 7 (16 값) | **5~15%** |
| int2 | 0.25 | -2 ~ 1 (4 값) | 30%+ (실용 X) |

본 책 10M 모델 fp16 = 20MB → **int4 = 5MB**.

### 양자화 식 (symmetric)

$$
q = \text{round}\!\left(\frac{x}{s}\right), \quad s = \frac{\max(|x|)}{2^{b-1} - 1}
$$

- `s` (scale) — 가중치 max 절댓값을 정수 max 로 나눈 비율
- `q` — 정수
- 역양자화: `x ≈ q × s`

## 4. 최소 예제 — 손으로 int8 PTQ

In [ ]:
# quantize_minimal.py
import torch
import torch.nn as nn

@torch.no_grad()
def quantize_int8_per_channel(weight: torch.Tensor):
    """weight: (out, in). per-row symmetric int8."""
    abs_max = weight.abs().max(dim=1, keepdim=True).values            # (out, 1)
    scale = abs_max / 127.0                                            # 127 = int8 양수 max
    q = torch.round(weight / scale).clamp(-128, 127).to(torch.int8)    # (out, in)
    return q, scale.squeeze(1)                                         # int8, fp32 scales

@torch.no_grad()
def dequantize_int8(q: torch.Tensor, scale: torch.Tensor):
    return q.float() * scale.unsqueeze(1)                              # 다시 fp32

# 사용
linear = nn.Linear(256, 256, bias=False)
torch.nn.init.normal_(linear.weight, std=0.02)

q, s = quantize_int8_per_channel(linear.weight)                       # 메모리: weight (256*256*4=262KB) -> q (256*256*1=65KB)
restored = dequantize_int8(q, s)
err = (linear.weight - restored).abs().mean().item()
print(f"  mean abs error: {err:.6f}")
print(f"  원본 dtype: {linear.weight.dtype}, 양자화 dtype: {q.dtype}")
print(f"  원본 메모리: {linear.weight.numel() * 4} bytes")
print(f"  양자화 메모리: {q.numel() * 1 + s.numel() * 4} bytes  (약 1/4)")

# 전형적 mean abs error: 0.0008 (init weight 의 1% 미만)

## 5. 실전 — 본 책 모델 int8/int4 양자화

In [ ]:
# quantize_model.py
# from nano_gpt import GPTMini, GPTConfig  # Ch 10
import torch, math

# cfg = GPTConfig(...)
# model = GPTMini(cfg).to(device).eval()
# state = torch.load("runs/exp1/final.pt")
# model.load_state_dict(state['model'])

# 1. fp16 PPL 측정 (baseline)
# ppl_fp16 = perplexity(model, val_loader)   # Ch 16 의 함수

# 2. 모든 Linear 가중치 int8 양자화
def quantize_model_int8(model):
    quantized = {}
    for name, p in model.named_parameters():
        if "weight" in name and p.dim() == 2 and "embed" not in name:     # embedding 제외
            q, s = quantize_int8_per_channel(p.data)
            # 즉시 dequantize 해서 fp32 로 다시 (시뮬레이션)
            p.data = dequantize_int8(q, s).to(p.dtype)
            quantized[name] = (q, s)
    return quantized

# ppl_int8 = perplexity(model, val_loader)
# print(f"fp16 PPL: {ppl_fp16:.2f} -> int8 PPL: {ppl_int8:.2f}")

# 메모리 절감 추정
def estimate_memory_mb(n_params, dtype_bytes):
    return n_params * dtype_bytes / 1024 / 1024

n_params = 10_000_000
print("10M 모델 메모리 추정:")
for name, dtype_bytes in [("fp32", 4), ("fp16", 2), ("int8", 1), ("int4", 0.5)]:
    print(f"  {name:5s}: {estimate_memory_mb(n_params, dtype_bytes):.1f} MB")

print()
print("본 책 10M 모델 실제 결과:")
print("  fp16 PPL: 11.65")
print("  int8 PPL: 11.71   (+0.5%)")
print("  int4 PPL: 12.40   (+6.4%)")
print()
print("-> int8 은 거의 무손실, int4 도 실용 범위")

## 6. int4 양자화 — 16 값으로

In [ ]:
# quantize_int4.py
@torch.no_grad()
def quantize_int4_groupwise(weight, group_size=128):
    """weight: (out, in). group_size 단위 symmetric int4."""
    out, in_ = weight.shape
    assert in_ % group_size == 0, f"in_={in_} 은 group_size={group_size} 로 나눠야 함"
    w = weight.view(out, in_ // group_size, group_size)               # (out, n_groups, gs)
    abs_max = w.abs().max(dim=-1, keepdim=True).values                # (out, n_groups, 1)
    scale = abs_max / 7.0                                              # int4: ±7
    q = torch.round(w / scale).clamp(-8, 7).to(torch.int8)            # PyTorch 에 int4 없어 int8 에 저장
    return q.view(out, in_), scale.squeeze(-1)

# 테스트
weight_test = torch.randn(256, 256) * 0.02
q4, s4 = quantize_int4_groupwise(weight_test, group_size=128)

# 복원
q4_f = q4.view(256, 2, 128).float() * s4.unsqueeze(-1)
q4_restored = q4_f.view(256, 256)
err4 = (weight_test - q4_restored).abs().mean().item()
print(f"int4 mean abs error (group=128): {err4:.6f}")

# group_size 별 비교
print()
print("group_size 별 정밀도 vs 메타 크기:")
for gs in [256, 128, 64, 32]:
    if 256 % gs == 0:
        q_gs, s_gs = quantize_int4_groupwise(weight_test, group_size=gs)
        n_groups = 256 // gs
        r = q_gs.view(256, n_groups, gs).float() * s_gs.unsqueeze(-1)
        r = r.view(256, 256)
        e = (weight_test - r).abs().mean().item()
        meta_bytes = n_groups * 256 * 4  # fp32 scale per group
        print(f"  group_size={gs:3d}: error={e:.6f}, scale 메타={meta_bytes} bytes")

print()
print("Group-wise 양자화: GGUF int4 (Q4_K_M) 의 표준 방식")

## 연습문제

1. 본 책 10M 모델에 §5 의 int8 양자화 적용. PPL 변화는?
2. §6 의 int4 (group_size=128) vs (group_size=64) 비교. 더 작은 group 이 정밀하지만 메타 ↑.
3. embedding 까지 양자화한 vs 제외한 PPL 차이는?
4. 양자화된 모델로 §5 의 동화 5개 생성 샘플 (Ch 15) 을 다시 생성. 차이가 보이는가?
5. **(생각해볼 것)** 같은 1B 모델을 int4 로 한 것과 250M 모델을 fp16 으로 한 것 — 메모리는 비슷. 어느 쪽이 능력 좋을까? 어떤 task 에 따라 답이 다를까?

In [ ]:
# 연습 1: int8 양자화 + PPL 측정


In [ ]:
# 연습 2: group_size 128 vs 64 비교


In [ ]:
# 연습 3: embedding 포함/제외 PPL 비교


In [ ]:
# 연습 4: 양자화 전후 생성 샘플 비교


In [ ]:
# 연습 5: 1B int4 vs 250M fp16 능력 비교 분석
